In [1]:
import os

os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

In [2]:
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments

d:\miniconda3\envs\PyTorch\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
ds = Dataset.load_from_disk("./nlpcc_2017/")
ds

Dataset({
    features: ['title', 'content'],
    num_rows: 5000
})

In [4]:
ds[0]

{'title': '澳大利亚央行将利率降至纪录低点,以应对疲软的经济前景,并遏制澳元进一步走强。',
 'content': '澳大利亚央行将利率降至纪录低点,以应对疲软的经济前景,并遏制澳元进一步走强。05/0513:37|评论(0)A+澳大利亚央行周二发布声明称,将关键利率由2.25%调降至2%,符合此前交易员及接受彭博调查的29位经济学家中25位的预期。据彭博社报道,上月澳央行官员曾警告,矿业之外的行业投资可能下滑。澳大利亚政府不太可能推出新的刺激措施,来扶助受本币升值和铁矿石价格下跌打击而低于潜在水平的经济增长。“鉴于大宗商品价格下跌,矿业投资还可能有低于当前预期的风险,”预计到降息的澳新银行高级经济学家FelicityEmmett在决议公布前编写的研究报告中称。他表示此次决议可能反映出“央行经济增长预估轨迹有所下调”。'}

In [5]:
ds = ds.train_test_split(500, seed=42)
ds

DatasetDict({
    train: Dataset({
        features: ['title', 'content'],
        num_rows: 4500
    })
    test: Dataset({
        features: ['title', 'content'],
        num_rows: 500
    })
})

In [6]:
tokenizer = AutoTokenizer.from_pretrained("Langboat/mengzi-t5-base")

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


In [7]:
def process_func(exmaples):
    contents = ["摘要生成: \n" + e for e in exmaples["content"]]
    inputs = tokenizer(contents, max_length=384, truncation=True)
    labels = tokenizer(text_target=exmaples["title"], max_length=64, truncation=True)
    inputs["labels"] = labels["input_ids"]
    return inputs

In [8]:
tokenized_ds = ds.map(process_func, batched=True)
tokenized_ds

Map: 100%|██████████| 500/500 [00:00<00:00, 5078.86 examples/s]


DatasetDict({
    train: Dataset({
        features: ['title', 'content', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 4500
    })
    test: Dataset({
        features: ['title', 'content', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 500
    })
})

In [9]:
tokenizer.decode(tokenized_ds["train"][0]["input_ids"])

'摘要生成: 新华网布鲁塞尔6月22日电(记者周珺孙奕)欧盟成员国外长会22日在卢森堡举行,会议宣布启动在地中海打击人口走私贩运的“欧盟地中海海军”行动计划。欧盟外交和安全政策高级代表莫盖里尼说:“欧盟从未如此重视移民问题,我们的目标是打击从移民的苦难中获益的商业模式。但这只是欧盟一个更广泛战略中的一部分。”她表示,欧盟将与非洲国家尤其是萨赫勒地区国家合作,并与国际移民组织和联合国难民署共同应对移民问题。“欧盟地中海海军”行动计划分为三个阶段,第一阶段重点监测和评估地中海人口走私和贩运网络;第二阶段将展开行动搜索和检查可疑船只;第三阶段将逮捕人口走私和贩运者,并对船只及相关资产进行处置。当天会后发布的公报指出,目前只是宣布启动第一阶段行动。考虑到联合国的授权和有关沿海国家的意见,欧盟将评估何时展开第二阶段行动。欧盟5月18日决定采取军事行动打击地中海人口走私活动。该行动的总部设在意大利罗马,欧盟初期为这一行动投入1182万欧元。联合国秘书长潘基文5月27日在布鲁塞尔表示,使用军事手段打击偷渡效果有限,需通盘考虑偷渡者的来源地、中转地、目的地等因素,以全面应对偷渡问题。</s>'

In [10]:
tokenizer.decode(tokenized_ds["train"][0]["labels"])

'欧盟宣布启动 “欧盟地中海海军”计划,打击地中海人口走私贩运,将与非洲国家、国际移民组织共同应对。</s>'

In [11]:
model = AutoModelForSeq2SeqLM.from_pretrained("Langboat/mengzi-t5-base")

In [13]:
import numpy as np
from rouge_chinese import Rouge

rouge = Rouge()

def compute_metric(evalPred):
    predictions, labels = evalPred
    decode_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decode_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    decode_preds = [" ".join(p) for p in decode_preds]
    decode_labels = [" ".join(l) for l in decode_labels]
    scores = rouge.get_scores(decode_preds, decode_labels, avg=True)
    return {
        "rouge-1": scores["rouge-1"]["f"],
        "rouge-2": scores["rouge-2"]["f"],
        "rouge-l": scores["rouge-l"]["f"],
    }

In [14]:
args = Seq2SeqTrainingArguments(
    output_dir="./summary",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=8,
    logging_steps=50,
    eval_strategy="steps",
    save_strategy="steps",
    metric_for_best_model="rouge-l",
    predict_with_generate=True
)

In [15]:
trainer = Seq2SeqTrainer(
    args=args,
    model=model,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["test"],
    compute_metrics=compute_metric,
    tokenizer=tokenizer,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer)
)

C:\Users\10433\AppData\Local\Temp\ipykernel_6996\1698183848.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


In [16]:
trainer.train()

Step,Training Loss,Validation Loss,Rouge-1,Rouge-2,Rouge-l
50,3.585900,2.532828,0.447205,0.272768,0.368381
100,2.701900,2.331373,0.467356,0.295321,0.385091
150,2.503600,2.243424,0.463902,0.295005,0.384697
200,2.188600,2.173399,0.469672,0.301714,0.390572
250,2.208100,2.118308,0.475455,0.303588,0.392089
300,2.066000,2.110950,0.480677,0.309870,0.399418
350,1.962000,2.083597,0.482544,0.311045,0.399709
400,1.957300,2.073592,0.480703,0.308359,0.398045


TrainOutput(global_step=423, training_loss=2.3738363730428347, metrics={'train_runtime': 742.9687, 'train_samples_per_second': 18.17, 'train_steps_per_second': 0.569, 'total_flos': 5491802113634304.0, 'train_loss': 2.3738363730428347, 'epoch': 3.0})

In [17]:
from transformers import pipeline


In [18]:
pipe = pipeline("text2text-generation", model=model, tokenizer=tokenizer, device=0)

Device set to use cuda:0


In [19]:
pipe("摘要生成:\n" + ds["test"][-1]["content"], max_length=64, do_sample=True)

Both `max_new_tokens` (=256) and `max_length`(=64) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': '临沂一男子因意外电话,将汽油泼到其身上,扬言纵火,扬言纵火,扬言纵火烧死,被刑3年。'}]

In [20]:
ds["test"][-1]["title"]

'临沂男子买汽油泼情敌,手持打火机扬言烧死对方,被巡逻民警及时制止;男子获刑3年'